# Explorador de Sesgos — LinkedIn Job Postings
**DataTalent Solutions S.L.** | Análisis y Visualización de Datos

Notebook interactivo: usa los controles de cada sección para explorar los datos desde distintos ángulos.

In [1]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import os, subprocess
    os.chdir('/content/drive/MyDrive/archive')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'plotly', 'ipywidgets', 'jinja2'])
else:
    import os
    print(f'Directorio: {os.getcwd()}')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

if IN_COLAB:
    pio.renderers.default = "colab"

COLS_DF = [
    'job_id', 'title', 'listed_time',
    'formatted_work_type', 'formatted_experience_level',
    'application_type', 'location', 'comp_country',
    'closed_time', 'sponsored', 'applies', 'salary_annual',
]

df     = pd.read_csv('data_roles_completo.csv', usecols=COLS_DF)
df_sal = pd.read_csv('data_roles_salario.csv',  usecols=['job_id', 'salary_annual'])

df['published_date'] = pd.to_datetime(df['listed_time'], unit='ms')
df['week']           = df['published_date'].dt.to_period('W').astype(str)
df['day_of_week']    = df['published_date'].dt.day_name()
df['month']          = df['published_date'].dt.to_period('M').astype(str)

if 'salary_annual' not in df.columns and 'salary_annual' in df_sal.columns:
    df = df.merge(df_sal[['job_id','salary_annual']].drop_duplicates('job_id'), on='job_id', how='left')

skills_map = pd.read_csv('mappings/skills.csv')
job_skills = pd.read_csv('jobs/job_skills.csv')

print(f'Dataset: {df.shape[0]:,} ofertas  |  {df.shape[1]} columnas')

Dataset: 19,725 ofertas  |  16 columnas


---
## Panel de navegación — Resumen de los 8 sesgos
Selecciona un sesgo para ver su ficha de impacto.

In [3]:
cards = {
    'Sesgo 1 — MNAR Salarial':          ('Datos faltantes no aleatorios', '71–95% nulos en columnas de salario', 'Sobreestima salarios del mercado real'),
    'Sesgo 2 — Geográfico':             ('Subrepresentación',             'Mayoría de ofertas en EEUU',          'Salarios y skills no aplicables a España'),
    'Sesgo 3 — Selección (LinkedIn)':   ('Fuente única',                  'Solo empresas activas en LinkedIn',   'Infravalora PYMEs y sector público'),
    'Sesgo 4 — Ausencia de género':     ('Atributo protegido ausente',    '0% de cobertura de género',           'Brecha salarial de género indetectable'),
    'Sesgo 5 — Temporal':               ('Cobertura temporal',            'Snapshot de ~abril 2024',             'Skills emergentes subrepresentadas'),
    'Sesgo 6 — Agregación de skills':   ('Medición',                      '35 categorías / +213k relaciones',    'Currículo de reskilling demasiado genérico'),
    'Sesgo 7 — Supervivencia':          ('Selección muestral',            '99%+ ofertas sin fecha de cierre',    'Mercado oculto (60-80%) invisible'),
    'Sesgo 8 — Applies subestimadas':   ('Medición parcial',              'Solo Easy Apply tiene applies',       'Correlación vistas↔solicitudes sesgada'),
}

fig_nav = go.Figure(data=[go.Table(
    columnwidth=[260, 210, 260, 290],
    header=dict(
        values=['<b>Sesgo</b>', '<b>Tipo</b>', '<b>Cuantificación</b>', '<b>Impacto clave</b>'],
        fill_color='#2c3e50',
        font=dict(color='white', size=12),
        align='left', height=36
    ),
    cells=dict(
        values=[
            list(cards.keys()),
            [v[0] for v in cards.values()],
            [v[1] for v in cards.values()],
            [v[2] for v in cards.values()],
        ],
        fill_color=[['#f8f9fa' if i % 2 == 0 else '#e9ecef' for i in range(len(cards))]]*4,
        font=dict(color='#212529', size=11),
        align='left', height=30
    )
)])
fig_nav.update_layout(
    title='Panel de navegación — Resumen de los 8 sesgos',
    margin=dict(t=60, l=10, r=10, b=10),
    height=340
)
display(fig_nav)

---
## Sesgo 1 — MNAR: ¿Qué grupos ocultan más los salarios?
Cambia la variable de agrupación para ver qué perfiles de oferta publican menos el salario.

In [4]:
group_options = {
    'Tipo de contrato':     'formatted_work_type',
    'Nivel de experiencia': 'formatted_experience_level',
    'Tipo de aplicación':   'application_type',
}
valid_options = {k: v for k, v in group_options.items() if v in df.columns}
keys = list(valid_options.keys())

traces = []
for key in keys:
    col = valid_options[key]
    rate = (
        df.groupby(col)['salary_annual']
        .apply(lambda x: round(x.notna().mean() * 100, 1))
        .reset_index()
    )
    rate.columns = [key, 'pct']
    rate = rate.sort_values('pct')
    traces.append(go.Bar(
        x=rate['pct'], y=rate[key],
        orientation='h',
        marker=dict(color=rate['pct'], colorscale='RdYlGn', cmin=0, cmax=100,
                    showscale=True, colorbar=dict(title='%', len=0.8)),
        text=[f'{v}%' for v in rate['pct']],
        textposition='outside',
        name=key,
        visible=(key == keys[0])
    ))

buttons = []
for i, key in enumerate(keys):
    vis = [j == i for j in range(len(keys))]
    buttons.append(dict(
        label=key, method='update',
        args=[{'visible': vis},
              {'title': f'MNAR — Tasa de publicación de salario por {key.lower()}',
               'xaxis.range': [0, 115]}]
    ))

fig_mnar = go.Figure(data=traces)
fig_mnar.update_layout(
    title=f'MNAR — Tasa de publicación de salario por {keys[0].lower()}',
    xaxis=dict(title='Salarios publicados (%)', range=[0, 115]),
    updatemenus=[dict(
        type='buttons', direction='left',
        x=0.0, y=1.15, xanchor='left', yanchor='top',
        showactive=True, bgcolor='#17a2b8',
        font=dict(color='white'), buttons=buttons
    )],
    margin=dict(t=100), height=400
)
display(fig_mnar)

**Interpretación:** El sesgo MNAR (Missing Not At Random) se confirma si la tasa de publicación varía significativamente entre grupos: las empresas no omiten el salario aleatoriamente, sino de forma estratégica. Un modelo predictivo entrenado solo con las ofertas que publican salario aprenderá de una muestra sesgada hacia arriba.

**Mitigación:** Usar mediana (no media) como referencia salarial y siempre reportar el % de datos usados.

---
## Sesgo 2 — Geográfico: ¿cuántas ubicaciones representan el mercado?
Mueve el slider para ver cuánto del dataset concentran las N ubicaciones más frecuentes.

In [5]:
geo_col = 'comp_country' if 'comp_country' in df.columns else 'location'
df['region'] = df[geo_col].str.split(',').str[-1].str.strip() if geo_col == 'location' else df[geo_col]
total_geo = df['region'].notna().sum()

n_values = list(range(5, 26))
traces, titles = [], []

for n in n_values:
    geo_data = df['region'].value_counts().head(n).reset_index()
    geo_data.columns = ['Ubicación', 'Ofertas']
    geo_data['pct'] = (geo_data['Ofertas'] / total_geo * 100).round(1)
    accumulated = geo_data['pct'].sum()
    geo_sorted = geo_data.sort_values('Ofertas')
    traces.append(go.Bar(
        x=geo_sorted['Ofertas'], y=geo_sorted['Ubicación'],
        orientation='h',
        text=[f'{p}%' for p in geo_sorted['pct']],
        textposition='outside',
        marker=dict(color=geo_sorted['Ofertas'].values, colorscale='Blues', showscale=False),
        showlegend=False,
        visible=(n == 10)
    ))
    titles.append(f'Top {n} ubicaciones — concentran el {accumulated:.1f}% del dataset')

default_idx = n_values.index(10)
steps = []
for i, (n, title) in enumerate(zip(n_values, titles)):
    vis = [j == i for j in range(len(n_values))]
    steps.append(dict(method='update', label=str(n),
                      args=[{'visible': vis}, {'title': title}]))

fig_geo = go.Figure(data=traces)
fig_geo.update_layout(
    title=titles[default_idx],
    xaxis_title='Ofertas',
    height=600,
    sliders=[dict(
        active=default_idx, steps=steps,
        currentvalue=dict(prefix='Top N: ', visible=True, font=dict(size=14)),
        pad=dict(t=50)
    )],
    margin=dict(t=80, b=130)
)
display(fig_geo)

**Interpretación:** Si las primeras 5 ubicaciones ya concentran más del 50% del dataset, el análisis representa principalmente esos mercados. Para DataTalent (mercado español) esto implica que los salarios de referencia están inflados respecto a la realidad local.

**Mitigación:** El notebook `Espana_Visualizacion.ipynb` complementa este análisis con datos del mercado español.

---
## Sesgo 3 — Selección: ¿qué tipo de empresas dominan LinkedIn?
Elige qué dimensión quieres explorar.

In [6]:
col_map = {
    'Tipo de aplicación':             'application_type',
    'Nivel de experiencia requerido': 'formatted_experience_level',
    'Tipo de contrato':               'formatted_work_type',
}
options = list(col_map.keys())

traces = []
for opt in options:
    col = col_map[opt]
    counts = df[col].value_counts().reset_index()
    counts.columns = [opt, 'Ofertas']
    counts['pct'] = (counts['Ofertas'] / len(df) * 100).round(1)
    traces.append(go.Bar(
        x=counts[opt], y=counts['Ofertas'],
        text=[f'{p}%' for p in counts['pct']],
        textposition='outside',
        marker=dict(color=counts['Ofertas'].values, colorscale='Purples', showscale=False),
        name=opt,
        visible=(opt == options[0])
    ))

buttons = []
for i, opt in enumerate(options):
    vis = [j == i for j in range(len(options))]
    buttons.append(dict(
        label=opt, method='update',
        args=[{'visible': vis},
              {'title': f'Sesgo de selección — Distribución por {opt.lower()}',
               'xaxis.title.text': opt}]
    ))

fig_sel = go.Figure(data=traces)
fig_sel.update_layout(
    title=f'Sesgo de selección — Distribución por {options[0].lower()}',
    xaxis_title=options[0],
    yaxis_title='Ofertas',
    updatemenus=[dict(
        type='buttons', direction='left',
        x=0.0, y=1.15, xanchor='left', yanchor='top',
        showactive=True, buttons=buttons
    )],
    margin=dict(t=100), height=450
)
display(fig_sel)

**Interpretación:** La distribución desequilibrada en cualquiera de estas dimensiones revela qué tipo de empresa domina el dataset. Si Full-time + grandes empresas acaparan el 80%, el modelo aprende principalmente de ese perfil e ignora el resto.

**Mitigación:** Diversificar fuentes de datos (Infojobs, Indeed, EURES).

---
## Sesgo 4 — Género: proxy por tipo de rol
Selecciona un tipo de rol para ver cómo se distribuye el salario en ese grupo.

In [7]:
df['role_type'] = 'Otro'
df.loc[df['title'].str.contains('engineer|developer|architect', case=False, na=False), 'role_type'] = 'Ingeniería / Desarrollo'
df.loc[df['title'].str.contains('analyst|analytics', case=False, na=False), 'role_type'] = 'Análisis de datos'
df.loc[df['title'].str.contains('scientist|science', case=False, na=False), 'role_type'] = 'Data Science'
df.loc[df['title'].str.contains('manager|director|lead', case=False, na=False), 'role_type'] = 'Gestión / Liderazgo'

available_roles = ['Todos'] + df['role_type'].unique().tolist()
src = df_sal if 'salary_annual' in df_sal.columns else df
global_median = src['salary_annual'].median()

traces, buttons = [], []
for i, role in enumerate(available_roles):
    if role == 'Todos':
        subset = src.dropna(subset=['salary_annual'])
        title = 'Distribución salarial — todos los roles'
    else:
        ids = df[df['role_type'] == role]['job_id']
        subset = src[src['job_id'].isin(ids)].dropna(subset=['salary_annual'])
        title = f'Distribución salarial — {role}'

    subset_filtered = subset[subset['salary_annual'] <= subset['salary_annual'].quantile(0.95)]

    if role != 'Todos':
        median_val = subset_filtered['salary_annual'].median()
        diff = (median_val - global_median) / global_median * 100
        title += f'<br><sup>Mediana: ${median_val:,.0f}  |  {diff:+.1f}% vs. mediana global</sup>'

    traces.append(go.Histogram(
        x=subset_filtered['salary_annual'],
        nbinsx=40, name=role,
        marker_color='#9b59b6',
        visible=(role == 'Todos')
    ))
    vis = [j == i for j in range(len(available_roles))]
    buttons.append(dict(label=role, method='update',
                        args=[{'visible': vis}, {'title': title}]))

fig_gender = go.Figure(data=traces)
fig_gender.update_layout(
    title='Distribución salarial — todos los roles',
    xaxis_title='Salario Anual (USD)',
    updatemenus=[dict(
        type='dropdown',
        x=0.0, y=1.15, xanchor='left', yanchor='top',
        buttons=buttons
    )],
    margin=dict(t=100), height=450
)
display(fig_gender)

**Interpretación:** Sin datos de género, usamos el tipo de rol como proxy. Si los roles históricamente más masculinizados (ingeniería) muestran medianas más altas, no podemos saber si es por el tipo de rol, la experiencia requerida, o por brecha de género.

**Mitigación:** Cruzar con Stack Overflow Developer Survey (incluye autodeclaración de género).

---
## Sesgo 5 — Temporal: ¿cómo se distribuyen las ofertas en el tiempo?
Cambia la granularidad para ver distintos patrones temporales.

In [8]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

option_defs = [
    ('Por semana',           'week',        'Ofertas publicadas por semana'),
    ('Por día de la semana', 'day_of_week', 'Ofertas publicadas por día de la semana'),
    ('Por mes',              'month',       'Ofertas publicadas por mes'),
]

traces, buttons = [], []
for i, (label, col, title) in enumerate(option_defs):
    group_data = df.groupby(col).size().reset_index(name='Ofertas')
    if col == 'day_of_week':
        group_data[col] = pd.Categorical(group_data[col], categories=day_order, ordered=True)
        group_data = group_data.sort_values(col)
    traces.append(go.Bar(
        x=group_data[col], y=group_data['Ofertas'],
        marker=dict(color=group_data['Ofertas'].values, colorscale='Blues', showscale=False),
        name=label,
        visible=(i == 0)
    ))
    vis = [j == i for j in range(len(option_defs))]
    buttons.append(dict(label=label, method='update',
                        args=[{'visible': vis}, {'title': title}]))

fig_temp = go.Figure(data=traces)
fig_temp.update_layout(
    title=option_defs[0][2],
    yaxis_title='Ofertas',
    xaxis=dict(tickangle=45),
    updatemenus=[dict(
        type='buttons', direction='left',
        x=0.0, y=1.15, xanchor='left', yanchor='top',
        showactive=True, bgcolor='#ffc107',
        font=dict(color='black'), buttons=buttons
    )],
    margin=dict(t=100), height=450
)
display(fig_temp)

**Interpretación:** Si los datos se concentran en pocas semanas, el dataset no captura variabilidad anual. El patrón por día de la semana revela si las empresas publican más en días laborables (lo esperado) — y cuantifica cuánto se inflarían las métricas si solo tomáramos datos de esos días.

**Mitigación:** Incorporar datasets de años anteriores para detectar tendencias y estacionalidad.

---
## Sesgo 6 — Agregación de Skills: ¿cuántas categorías necesitas para cubrir el mercado?
Ajusta el slider para ver qué porcentaje del mercado cubres con las N skills más demandadas.

In [9]:
skill_dist = job_skills['skill_abr'].value_counts().reset_index()
skill_dist.columns = ['skill_abr', 'num_ofertas']
skill_dist = skill_dist.merge(skills_map, on='skill_abr', how='left')
skill_dist['skill_name'] = skill_dist['skill_name'].fillna(skill_dist['skill_abr'])
total_skills = skill_dist['num_ofertas'].sum()

n_values = [1, 5, 10, 15, 20, 25, 30, 35]
traces, titles = [], []

for n in n_values:
    top = skill_dist.head(n).copy()
    coverage = top['num_ofertas'].sum() / total_skills * 100
    top_sorted = top.sort_values('num_ofertas')
    traces.append(go.Bar(
        x=top_sorted['num_ofertas'], y=top_sorted['skill_name'],
        orientation='h',
        marker=dict(color=top_sorted['num_ofertas'].values, colorscale='Viridis', showscale=False),
        showlegend=False,
        visible=(n == 10)
    ))
    titles.append(
        f'Top {n} skills → cubren el {coverage:.1f}% de todas las menciones'
        f'<br><sup>Las {35 - n} restantes cubren el {100 - coverage:.1f}% — skills de nicho</sup>'
    )

default_idx = n_values.index(10)
steps = []
for i, (n, title) in enumerate(zip(n_values, titles)):
    vis = [j == i for j in range(len(n_values))]
    steps.append(dict(method='update', label=str(n),
                      args=[{'visible': vis}, {'title': title}]))

fig_skills = go.Figure(data=traces)
fig_skills.update_layout(
    title=titles[default_idx],
    xaxis_title='Menciones en ofertas',
    yaxis_title='Categoría',
    height=600,
    sliders=[dict(
        active=default_idx, steps=steps,
        currentvalue=dict(prefix='Top N skills: ', visible=True, font=dict(size=14)),
        pad=dict(t=50)
    )],
    margin=dict(t=100, b=130)
)
display(fig_skills)

**Interpretación:** Desplaza el slider hasta encontrar el punto donde la cobertura sube poco con cada nueva categoría añadida — ese es el número mínimo de skills que DataTalent debería priorizar en su programa de reskilling. El sesgo de agregación hace que cada una de estas 35 categorías esconda múltiples skills concretas (Python, SQL, Spark, etc.).

**Mitigación:** Analizar el texto libre de `description` con NLP para extraer skills específicas.

---
## Sesgo 7 — Supervivencia: ¿cuánto del mercado real estamos viendo?
Explora las ofertas patrocinadas vs orgánicas y qué tipo de empresa las publica.

In [10]:
# Trace 0 — pie: patrocinadas vs orgánicas
pat_data = df['sponsored'].map({0: 'Orgánica', 1: 'Patrocinada'}).value_counts()
trace_pat = go.Pie(
    labels=pat_data.index, values=pat_data.values, hole=0.4,
    marker=dict(colors=['#3498db','#e67e22']),
    textinfo='percent+label+value', name='Patrocinadas',
    visible=True, domain=dict(x=[0.2, 0.8], y=[0.1, 0.9])
)

# Trace 1 — pie: con/sin fecha de cierre
closed_count = df['closed_time'].notna().sum()
trace_close = go.Pie(
    labels=['Con fecha de cierre', 'Sin fecha de cierre'],
    values=[closed_count, len(df) - closed_count], hole=0.4,
    marker=dict(colors=['#2ecc71','#e74c3c']),
    textinfo='percent+label+value', name='Cierre',
    visible=False, domain=dict(x=[0.2, 0.8], y=[0.1, 0.9])
)

# Trace 2 — bar: tipo de aplicación
app_counts = df['application_type'].value_counts().reset_index()
app_counts.columns = ['Tipo', 'Ofertas']
app_counts['pct'] = (app_counts['Ofertas'] / len(df) * 100).round(1)
trace_app = go.Bar(
    x=app_counts['Tipo'], y=app_counts['Ofertas'],
    text=[f'{p}%' for p in app_counts['pct']],
    textposition='outside',
    marker=dict(color=app_counts['Ofertas'].values, colorscale='Purples', showscale=False),
    name='Tipo aplicación', visible=False
)

option_titles = [
    ('Patrocinadas vs orgánicas',       'Ofertas patrocinadas vs orgánicas'),
    ('Ofertas con/sin fecha de cierre', '¿Cuántas ofertas registran su cierre? (proxy de mercado visible)'),
    ('Tipo de aplicación',              'Tipo de aplicación — mayoría redirige fuera de LinkedIn'),
]

buttons = []
for i, (label, title) in enumerate(option_titles):
    vis = [j == i for j in range(3)]
    show_axes = (i == 2)
    buttons.append(dict(
        label=label, method='update',
        args=[{'visible': vis},
              {'title': title, 'xaxis.visible': show_axes, 'yaxis.visible': show_axes}]
    ))

fig_surv = go.Figure(data=[trace_pat, trace_close, trace_app])
fig_surv.update_layout(
    title=option_titles[0][1],
    xaxis=dict(visible=False), yaxis=dict(visible=False),
    updatemenus=[dict(
        type='buttons', direction='left',
        x=0.0, y=1.15, xanchor='left', yanchor='top',
        showactive=True, buttons=buttons
    )],
    margin=dict(t=100), height=450
)
display(fig_surv)

**Interpretación:** El porcentaje de ofertas sin fecha de cierre confirma que el dataset es una foto puntual, no un registro completo. Las ofertas patrocinadas tienen más visibilidad pero pueden no ser representativas del mercado — las empresas que pagan por visibilidad tienen un perfil distinto.

**Mitigación:** Encuestas a candidatos para estimar el peso del mercado oculto.

---
## Sesgo 8 — Applies Subestimadas: el contador solo captura Easy Apply
Ajusta el percentil máximo para filtrar outliers y ver cómo cambia la distribución de solicitudes.

In [11]:
df_applies = df[['application_type', 'applies']].dropna(subset=['applies'])
offsite_pct = (df['application_type'] == 'OffsiteApply').mean() * 100
print(f'OffsiteApply (applies = NaN): {offsite_pct:.1f}% de las ofertas')
print(f'Solo el {100 - offsite_pct:.1f}% tiene datos reales de applies')

pct_values = [75, 80, 85, 90, 95, 99]
traces, titles = [], []

for pct in pct_values:
    threshold = df_applies['applies'].quantile(pct / 100)
    df_filtered = df_applies[df_applies['applies'] <= threshold]
    traces.append(go.Box(
        x=df_filtered['application_type'],
        y=df_filtered['applies'],
        name=f'p{pct}',
        visible=(pct == 95)
    ))
    titles.append(
        f'Applies por tipo de aplicación (p{pct}, umbral: {threshold:.0f})'
        f'<br><sup>{len(df_filtered):,} ofertas mostradas de {len(df_applies):,} | '
        f'{offsite_pct:.1f}% OffsiteApply sin datos</sup>'
    )

default_idx = pct_values.index(95)
steps = []
for i, (pct, title) in enumerate(zip(pct_values, titles)):
    vis = [j == i for j in range(len(pct_values))]
    steps.append(dict(method='update', label=f'p{pct}',
                      args=[{'visible': vis}, {'title': title}]))

fig_applies = go.Figure(data=traces)
fig_applies.update_layout(
    title=titles[default_idx],
    xaxis_title='Tipo de aplicación',
    yaxis_title='Solicitudes registradas',
    height=500,
    sliders=[dict(
        active=default_idx, steps=steps,
        currentvalue=dict(prefix='Percentil máx: ', visible=True, font=dict(size=14)),
        pad=dict(t=50)
    )],
    margin=dict(t=120, b=130)
)
display(fig_applies)

OffsiteApply (applies = NaN): 61.0% de las ofertas
Solo el 39.0% tiene datos reales de applies


**Interpretación:** Al bajar el percentil máximo puedes ver cómo los outliers extremos distorsionan la distribución. El sesgo principal es estructural: la mayoría de las grandes empresas usa portales externos (OffsiteApply) y sus solicitudes son completamente invisibles en este dataset.

**Mitigación:** Usar `views` como proxy de interés para todas las ofertas — está disponible independientemente del tipo de aplicación.

---
## Tabla resumen final

In [12]:
summary = pd.DataFrame([
    (1, 'MNAR Salarial',           'Datos faltantes no aleatorios', '71–95% nulos',               'Sobreestima salarios',                'Usar mediana + fuentes externas'),
    (2, 'Geográfico',              'Subrepresentación',             'Mayoría EEUU',                'Salarios/skills no aplican a España', 'Infojobs / SEPE'),
    (3, 'Selección (LinkedIn)',    'Fuente única',                  'Solo marca empleadora fuerte','Infravalora PYMEs',                   'Diversificar fuentes'),
    (4, 'Ausencia de género',      'Atributo protegido ausente',    '0% cobertura de género',      'Brecha de género indetectable',       'Stack Overflow Survey'),
    (5, 'Temporal',                'Cobertura temporal',            'Snapshot ~abril 2024',        'Skills emergentes subrepresentadas',  'Series históricas 2022–2024'),
    (6, 'Agregación de skills',    'Medición',                      '35 cats / 213k relaciones',   'Currículo demasiado genérico',        'NLP en descripciones'),
    (7, 'Supervivencia',           'Selección muestral',            '99%+ sin fecha de cierre',    'Mercado oculto invisible',            'Encuestas a candidatos'),
    (8, 'Applies subestimadas',    'Medición parcial',              'Solo Easy Apply tiene datos', 'Correlación vistas↔applies sesgada',  'Usar views como proxy'),
], columns=['#','Sesgo','Tipo','Cuantificación','Impacto','Mitigación'])

summary.set_index('#', inplace=True)
summary.style.set_properties(**{'text-align': 'left'}).set_table_styles(
    [{'selector': 'th', 'props': [('text-align', 'left')]}]
)

,Sesgo,Tipo,Cuantificación,Impacto,Mitigación
#,,,,,
1,MNAR Salarial,Datos faltantes no aleatorios,71–95% nulos,Sobreestima salarios,Usar mediana + fuentes externas
2,Geográfico,Subrepresentación,Mayoría EEUU,Salarios/skills no aplican a España,Infojobs / SEPE
3,Selección (LinkedIn),Fuente única,Solo marca empleadora fuerte,Infravalora PYMEs,Diversificar fuentes
4,Ausencia de género,Atributo protegido ausente,0% cobertura de género,Brecha de género indetectable,Stack Overflow Survey
5,Temporal,Cobertura temporal,Snapshot ~abril 2024,Skills emergentes subrepresentadas,Series históricas 2022–2024
6,Agregación de skills,Medición,35 cats / 213k relaciones,Currículo demasiado genérico,NLP en descripciones
7,Supervivencia,Selección muestral,99%+ sin fecha de cierre,Mercado oculto invisible,Encuestas a candidatos
8,Applies subestimadas,Medición parcial,Solo Easy Apply tiene datos,Correlación vistas↔applies sesgada,Usar views como proxy
